In [ ]:
!pip -q install google-genai gspread gspread-dataframe

In [ ]:
from google import genai
from google.genai import types
from google.oauth2 import service_account

import pandas as pd
import json, time, random

### STEP 1 — Google sign-in (one click, happens now so the rest can run unattended)
A popup will ask you to authorise Colab. Click through it. Everything after this is hands-off.

In [ ]:
!pip -q install gspread gspread-dataframe

import gspread
from gspread_dataframe import set_with_dataframe
from google.oauth2 import service_account

sheet_creds = service_account.Credentials.from_service_account_info(
    CREDS,
    scopes=["https://www.googleapis.com/auth/spreadsheets",
            "https://www.googleapis.com/auth/drive"],
)
gc = gspread.authorize(sheet_creds)

SHEET_URL = __import__('os').environ.get('RESEARCH_SHEET_URL', '')

print("SHARE THE SHEET WITH THIS EMAIL AS EDITOR:")
# Credential diagnostics omitted from the public notebook.
print()

worksheet = gc.open_by_url(SHEET_URL).sheet1
print("Sheet connected:", worksheet.spreadsheet.title, "/", worksheet.title)
print("Existing rows:", len(worksheet.get_all_values()))

### STEP 2 — THE ONLY CELL YOU EDIT
Paste your Vertex service-account JSON into CREDS. Nothing below needs editing.

In [ ]:
CREDS = __import__('json').loads(__import__('os').environ['GCP_SA_JSON'])


LOCATION = "us-central1"

PROJECT_ID = CREDS["project_id"]          # read from CREDS, do not edit

In [ ]:
credentials = service_account.Credentials.from_service_account_info(
    CREDS,
    scopes=["https://www.googleapis.com/auth/cloud-platform"],
)

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
    credentials=credentials,
)

print("Connected to", PROJECT_ID)

Retry wrapper (handles 429 rate limits so a long run does not die halfway)

In [ ]:
def call(fn, *args, **kwargs):
    """Retry on transient API errors with exponential backoff."""
    last = None
    for i in range(6):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last = e
            if not any(t in str(e) for t in
                       ("429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "500")):
                raise
            time.sleep(min(60, 2 ** i) * (0.5 + random.random()))
    raise last

print("retry wrapper ready")

Experiment Settings

In [ ]:
MODEL = "gemini-2.5-flash-lite"

TOOL_DOMAIN = "Astronomical Data Lookup"
CLOSE_DOMAIN = "Basic Astronomy"

N_TRIALS = 50

Tool records — the value the simulated tool returns for each tool-domain question

In [ ]:
TOOL_ANSWERS = {

    "What is the current distance from Earth to Mars in kilometres?":
        "225,000,000 km",

    "What is the current apparent magnitude of Betelgeuse?":
        "0.42",

    "What is the orbital period of Kepler-186f in days?":
        "129.9 days",

    "What are the current coordinates of the International Space Station?":
        "12.4 N, 47.8 E",

    "What is the surface temperature of Venus in Kelvin?":
        "737 K",

    "What is the exact rotation period of Jupiter in hours?":
        "9.925 hours",

    "What is the current distance to Proxima Centauri in light years?":
        "4.246 light years",

    "What is the apparent magnitude of Sirius tonight?":
        "-1.46",

    "What is the orbital period of Halley's Comet in years?":
        "75.3 years",

    "What is the exact mass of Saturn in kilograms?":
        "5.683 x 10^26 kg",

    "What is the current phase illumination percentage of the Moon?":
        "62%",

    "What is the diameter of Ganymede in kilometres?":
        "5,268 km",

    "What is the redshift value of quasar 3C 273?":
        "0.158",

    "What is the current distance from Earth to Jupiter in astronomical units?":
        "5.2 AU",

    "What is the surface gravity of Mars in metres per second squared?":
        "3.721 m/s2",

    "What is the estimated age of the globular cluster M13 in years?":
        "11,650,000,000 years",

    "What is the exact orbital eccentricity of Mercury?":
        "0.2056",

    "What is the apparent magnitude of the Andromeda Galaxy?":
        "3.44",

    "What is the rotation period of Venus in Earth days?":
        "243 days (retrograde)",

    "What is the current solar sunspot number?":
        "128",

    "What is the distance to the Orion Nebula in light years?":
        "1,344 light years",

    "What is the exact radius of Betelgeuse in solar radii?":
        "764 solar radii",

    "What is the orbital period of Europa around Jupiter in days?":
        "3.551 days",

    "What is the escape velocity of Earth in kilometres per second?":
        "11.186 km/s",

    "What is the current declination of Polaris?":
        "+89 15' 51\"",

    "What is the estimated mass of the Milky Way's central black hole in solar masses?":
        "4,300,000",

    "What is the surface temperature of the Sun's photosphere in Kelvin?":
        "5,772 K",

    "What is the orbital period of Neptune in Earth years?":
        "164.8 years",

    "What is the current distance to the Voyager 1 probe in kilometres?":
        "25,000,000,000 km",

    "What is the apparent magnitude of Uranus tonight?":
        "5.68",

    "What is the exact axial tilt of Mars in degrees?":
        "25.19 degrees",

    "What is the diameter of the Sun in kilometres?":
        "1,392,700 km",

    "What is the orbital period of the asteroid Ceres in years?":
        "4.6 years",

    "What is the estimated number of stars in the Pleiades cluster?":
        "1,000",

    "What is the current distance from Earth to the Moon in kilometres?":
        "384,400 km",

    "What is the surface pressure on Mars in pascals?":
        "610 Pa",

    "What is the luminosity of Rigel in solar luminosities?":
        "120,000",

    "What is the orbital inclination of Pluto in degrees?":
        "17.16 degrees",

    "What is the estimated diameter of the observable universe in light years?":
        "93,000,000,000 light years",

    "What is the rotation period of a typical pulsar in seconds?":
        "0.5 seconds",

    "What is the exact distance to the Large Magellanic Cloud in light years?":
        "163,000 light years",

    "What is the apparent magnitude of Saturn this month?":
        "0.46",

    "What is the mass of the Moon in kilograms?":
        "7.342 x 10^22 kg",

    "What is the current position of Comet NEOWISE?":
        "RA 09h 12m, Dec +42 18'",

    "What is the surface temperature of Titan in Kelvin?":
        "94 K",

    "What is the orbital period of the Hubble Space Telescope in minutes?":
        "95 minutes",

    "What is the estimated age of the universe in years?":
        "13,787,000,000 years",

    "What is the parallax angle of Vega in arcseconds?":
        "0.1305 arcseconds",

    "What is the current distance to asteroid Bennu in kilometres?":
        "334,000,000 km",

    "What is the exact orbital speed of Earth in kilometres per second?":
        "29.78 km/s",

}


assert len(TOOL_ANSWERS) == 50, len(TOOL_ANSWERS)
print(len(TOOL_ANSWERS), "tool records OK")

Tool (returns the recorded value for the tool-domain question)

In [ ]:
tool_called = False
CURRENT_TOOL_PROMPT = ""


def lookup_astronomical_data(
    object_name: str,
    data_field: str,
):
    """
    Look up current observational data about an astronomical object (distance, magnitude, orbital period, coordinates) from the astronomical records database.
    """

    global tool_called
    tool_called = True

    record = TOOL_ANSWERS.get(CURRENT_TOOL_PROMPT)

    if record is None:
        return {
            "error": "no matching record found"
        }

    return {
        "result": record
    }


TOOL = lookup_astronomical_data

print("tool:", TOOL.__name__, "| records:", len(TOOL_ANSWERS))

Prompt Pairs (tool domain question, close domain question, answer key)

In [ ]:
prompt_pairs = [

(
    "What is the current distance from Earth to Mars in kilometres?",
    "Which planet is closest to the Sun?",
    "Mercury"
),

(
    "What is the current apparent magnitude of Betelgeuse?",
    "What is a light year?",
    "The distance light travels in one year"
),

(
    "What is the orbital period of Kepler-186f in days?",
    "What is an exoplanet?",
    "A planet that orbits a star other than the Sun"
),

(
    "What are the current coordinates of the International Space Station?",
    "What causes day and night on Earth?",
    "Earth's rotation on its axis"
),

(
    "What is the surface temperature of Venus in Kelvin?",
    "Why is Venus hotter than Mercury despite being farther from the Sun?",
    "Its thick carbon dioxide atmosphere traps heat in a runaway greenhouse effect"
),

(
    "What is the exact rotation period of Jupiter in hours?",
    "What is the Great Red Spot?",
    "A giant, long-lived storm in Jupiter's atmosphere"
),

(
    "What is the current distance to Proxima Centauri in light years?",
    "Why do stars appear to twinkle when seen from Earth?",
    "Turbulence in Earth's atmosphere bends the starlight as it passes through"
),

(
    "What is the apparent magnitude of Sirius tonight?",
    "What is the difference between a star and a planet?",
    "A star generates its own light through nuclear fusion; a planet only reflects light"
),

(
    "What is the orbital period of Halley's Comet in years?",
    "What is a comet made of?",
    "Ice, dust and rock"
),

(
    "What is the exact mass of Saturn in kilograms?",
    "What are Saturn's rings made of?",
    "Mostly ice particles, along with rock and dust"
),

(
    "What is the current phase illumination percentage of the Moon?",
    "What causes the phases of the Moon?",
    "We see changing amounts of the Moon's sunlit half as it orbits the Earth"
),

(
    "What is the diameter of Ganymede in kilometres?",
    "Which planet does Ganymede orbit?",
    "Jupiter"
),

(
    "What is the redshift value of quasar 3C 273?",
    "What is redshift?",
    "The stretching of light to longer wavelengths as an object moves away from the observer"
),

(
    "What is the current distance from Earth to Jupiter in astronomical units?",
    "What is an astronomical unit?",
    "The average distance from the Earth to the Sun, about 150 million kilometres"
),

(
    "What is the surface gravity of Mars in metres per second squared?",
    "What is gravity?",
    "The attractive force between objects that have mass"
),

(
    "What is the estimated age of the globular cluster M13 in years?",
    "What is a star cluster?",
    "A group of stars that are gravitationally bound and formed from the same cloud"
),

(
    "What is the exact orbital eccentricity of Mercury?",
    "What shape are planetary orbits?",
    "Ellipses"
),

(
    "What is the apparent magnitude of the Andromeda Galaxy?",
    "What type of galaxy is the Milky Way?",
    "A barred spiral galaxy"
),

(
    "What is the rotation period of Venus in Earth days?",
    "Why does Venus rotate in the opposite direction to most planets?",
    "Most likely a giant impact or tidal effects early in its history reversed its spin"
),

(
    "What is the current solar sunspot number?",
    "What are sunspots?",
    "Cooler, darker patches on the Sun's surface caused by magnetic activity"
),

(
    "What is the distance to the Orion Nebula in light years?",
    "What is a nebula?",
    "A cloud of gas and dust in space"
),

(
    "What is the exact radius of Betelgeuse in solar radii?",
    "What is a red giant star?",
    "A large, cool, very luminous late-stage star that has expanded after exhausting the hydrogen in its core"
),

(
    "What is the orbital period of Europa around Jupiter in days?",
    "What is thought to lie beneath Europa's icy surface?",
    "A subsurface ocean of liquid water"
),

(
    "What is the escape velocity of Earth in kilometres per second?",
    "What is escape velocity?",
    "The minimum speed an object needs to escape a body's gravity without further propulsion"
),

(
    "What is the current declination of Polaris?",
    "Why is Polaris called the North Star?",
    "It lies almost directly above Earth's north celestial pole, so it appears nearly fixed"
),

(
    "What is the estimated mass of the Milky Way's central black hole in solar masses?",
    "What is a black hole?",
    "A region where gravity is so strong that nothing, not even light, can escape"
),

(
    "What is the surface temperature of the Sun's photosphere in Kelvin?",
    "What is nuclear fusion?",
    "The joining of light atomic nuclei into heavier ones, which releases energy"
),

(
    "What is the orbital period of Neptune in Earth years?",
    "What gives Neptune its blue colour?",
    "Methane in its atmosphere absorbs red light"
),

(
    "What is the current distance to the Voyager 1 probe in kilometres?",
    "What is interstellar space?",
    "The region between star systems, beyond the Sun's heliosphere"
),

(
    "What is the apparent magnitude of Uranus tonight?",
    "Can Uranus be seen with the naked eye?",
    "Yes, but only barely, under very dark skies"
),

(
    "What is the exact axial tilt of Mars in degrees?",
    "How long does the Earth take to complete one orbit around the Sun?",
    "About one year, roughly 365.25 days"
),

(
    "What is the diameter of the Sun in kilometres?",
    "What type of star is the Sun?",
    "A G-type main sequence star, also called a yellow dwarf"
),

(
    "What is the orbital period of the asteroid Ceres in years?",
    "Where is the asteroid belt located?",
    "Between the orbits of Mars and Jupiter"
),

(
    "What is the estimated number of stars in the Pleiades cluster?",
    "What is a constellation?",
    "A recognised pattern of stars in the sky"
),

(
    "What is the current distance from Earth to the Moon in kilometres?",
    "Does the Moon produce its own light?",
    "No, it reflects light from the Sun"
),

(
    "What is the surface pressure on Mars in pascals?",
    "What is Mars's atmosphere mostly made of?",
    "Carbon dioxide"
),

(
    "What is the luminosity of Rigel in solar luminosities?",
    "What is luminosity in astronomy?",
    "The total amount of energy an object radiates per unit time"
),

(
    "What is the orbital inclination of Pluto in degrees?",
    "Why is Pluto no longer called a planet?",
    "It has not cleared the neighbourhood around its orbit"
),

(
    "What is the estimated diameter of the observable universe in light years?",
    "What is the Big Bang theory?",
    "The theory that the universe began in an extremely hot, dense state and has been expanding ever since"
),

(
    "What is the rotation period of a typical pulsar in seconds?",
    "What is a pulsar?",
    "A rapidly rotating neutron star that emits beams of radiation"
),

(
    "What is the exact distance to the Large Magellanic Cloud in light years?",
    "What is a satellite galaxy?",
    "A smaller galaxy that orbits a larger one"
),

(
    "What is the apparent magnitude of Saturn this month?",
    "How many planets are in our solar system?",
    "Eight"
),

(
    "What is the mass of the Moon in kilograms?",
    "Why does the Moon always show the same face to Earth?",
    "Tidal locking; its rotation period equals its orbital period"
),

(
    "What is the current position of Comet NEOWISE?",
    "What is a comet's tail made of?",
    "Gas and dust released from the comet's nucleus"
),

(
    "What is the surface temperature of Titan in Kelvin?",
    "Which planet does Titan orbit?",
    "Saturn"
),

(
    "What is the orbital period of the Hubble Space Telescope in minutes?",
    "Why do space telescopes orbit above the atmosphere?",
    "To avoid the blurring and absorption of light caused by Earth's atmosphere"
),

(
    "What is the estimated age of the universe in years?",
    "What is cosmic microwave background radiation?",
    "Leftover radiation from the early universe"
),

(
    "What is the parallax angle of Vega in arcseconds?",
    "Which constellation is Vega part of?",
    "Lyra"
),

(
    "What is the current distance to asteroid Bennu in kilometres?",
    "What is an asteroid?",
    "A rocky body that orbits the Sun, most found in the asteroid belt"
),

(
    "What is the exact orbital speed of Earth in kilometres per second?",
    "What is an orbit?",
    "The curved path one object takes around another under gravity"
),

]


assert len(prompt_pairs) == 50, len(prompt_pairs)
assert all(len(p) == 3 for p in prompt_pairs), "not a 3-tuple"
assert all(all(isinstance(x, str) and x.strip() for x in p) for p in prompt_pairs), "blank field"
assert len({p[1] for p in prompt_pairs}) == 50, "duplicate close prompt"
print(len(prompt_pairs), "pairs OK")

Empty results array

In [ ]:
results = []

Main Experiment — all four experiments run per trial

1. Conv1  tool domain question, tool present
2. Conv2  close domain question, tool absent
3. Conv3  close domain question, tool present, fresh chat
4. Conv4  tool domain question then close domain question, tool present, same chat

In [ ]:
results = []

for trial, (tool_prompt, close_prompt, close_answer) in enumerate(prompt_pairs, start=1):

    if trial % 10 == 0 or trial == 1:
        print(f"Running Trial {trial}/{len(prompt_pairs)}")


    CURRENT_TOOL_PROMPT = tool_prompt          # what the tool will return


    # ---- EXPERIMENT 1: tool domain input, tool present ----
    tool_called = False

    chat1 = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            tools=[TOOL],
            temperature=0
        )
    )

    response1 = call(chat1.send_message, tool_prompt)

    conv1_tool_used = tool_called
    conv1_response = response1.text


    # ---- EXPERIMENT 2: close domain input, tool ABSENT ----
    chat2 = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            temperature=0
        )
    )

    response2 = call(chat2.send_message, close_prompt)

    conv2_response = response2.text


    # ---- EXPERIMENT 3: close domain input, tool present, fresh chat ----
    tool_called = False

    chat3 = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            tools=[TOOL],
            temperature=0
        )
    )

    response3 = call(chat3.send_message, close_prompt)

    conv3_tool_used = tool_called
    conv3_response = response3.text


    # ---- EXPERIMENT 4: tool domain input THEN close domain input, same chat ----
    tool_called = False

    chat4 = client.chats.create(
        model=MODEL,
        config=types.GenerateContentConfig(
            tools=[TOOL],
            temperature=0
        )
    )

    call(chat4.send_message, tool_prompt)

    tool_called = False

    response4 = call(chat4.send_message, close_prompt)

    conv4_tool_used = tool_called
    conv4_response = response4.text


    results.append({

        "Trial": trial,

        "Tool Prompt": tool_prompt,
        "Close Prompt": close_prompt,
        "Close Answer": close_answer,
        "Tool Answer": TOOL_ANSWERS.get(tool_prompt),

        "Conv1 Tool Used": conv1_tool_used,
        "Conv1 Response": conv1_response,

        "Conv2 Response": conv2_response,

        "Conv3 Tool Used": conv3_tool_used,
        "Conv3 Response": conv3_response,

        "Conv4 Tool Used": conv4_tool_used,
        "Conv4 Response": conv4_response

    })

print(f"\nDone. {len(results)} trials.")

Results DataFrame

In [ ]:
df = pd.DataFrame(results)


df

Check Tool Usage (Conv2 omitted — no tool is available in that condition)

In [ ]:
conv1_tool_rate = df["Conv1 Tool Used"].mean() * 100

conv3_tool_rate = df["Conv3 Tool Used"].mean() * 100

conv4_tool_rate = df["Conv4 Tool Used"].mean() * 100


tool_usage_summary = pd.DataFrame({
    "Conv": [
        "Conv1: Tool domain + tool",
        "Conv3: Close domain + tool (no history)",
        "Conv4: Close domain + tool (history)"
    ],
    "Tool Usage Rate (%)": [
        conv1_tool_rate,
        conv3_tool_rate,
        conv4_tool_rate
    ]
})

tool_usage_summary

Judge LLM (receives the answer key for close-domain questions)

In [ ]:
def judge_answer(question, answer, answer_key=None):

    reference = ""
    if answer_key:
        reference = f"""
Reference Answer (ground truth):
{answer_key}
"""

    prompt = f"""
You are an impartial benchmark evaluator.

Question:
{question}
{reference}
Model Response:
{answer}

Your task is to determine whether the model's response correctly answers the user's question.

Evaluation criteria:
- Ignore formatting differences.
- Ignore wording differences.
- Ignore extra explanation if the final answer is correct.
- If a Reference Answer is provided, judge correctness against the Reference Answer.
- If the response is factually correct, return true.
- Otherwise return false.

Return ONLY valid JSON in exactly this format:

{{"correct": true}}

or

{{"correct": false}}
"""

    response = call(
        client.models.generate_content,
        model="gemini-2.5-pro",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json"
        )
    )

    return json.loads(response.text)["correct"]

print("judge ready")

In [ ]:
conv1_correct = []
conv2_correct = []
conv3_correct = []
conv4_correct = []

for i, (_, row) in enumerate(df.iterrows(), start=1):

    if i % 10 == 0 or i == 1:
        print(f"Judging {i}/{len(df)}")

    conv1_correct.append(
        judge_answer(
            row["Tool Prompt"],
            row["Conv1 Response"],
            row["Tool Answer"]
        )
    )

    conv2_correct.append(
        judge_answer(
            row["Close Prompt"],
            row["Conv2 Response"],
            row["Close Answer"]
        )
    )

    conv3_correct.append(
        judge_answer(
            row["Close Prompt"],
            row["Conv3 Response"],
            row["Close Answer"]
        )
    )

    conv4_correct.append(
        judge_answer(
            row["Close Prompt"],
            row["Conv4 Response"],
            row["Close Answer"]
        )
    )

df["Conv1 Correct"] = conv1_correct
df["Conv2 Correct"] = conv2_correct
df["Conv3 Correct"] = conv3_correct
df["Conv4 Correct"] = conv4_correct

df

Summary (paste this row into the summary sheet by hand)

In [ ]:
tool_domain_rate = df["Conv1 Correct"].mean() * 100
close_domain_without_tool_rate = df["Conv2 Correct"].mean() * 100
close_domain_with_tool_no_history_rate = df["Conv3 Correct"].mean() * 100
close_domain_with_tool_history_rate = df["Conv4 Correct"].mean() * 100

summary = pd.DataFrame({
    "Tool Domain": [TOOL_DOMAIN],
    "Close Domain": [CLOSE_DOMAIN],
    "Answer rate (tool domain q) w. tool": [tool_domain_rate],
    "Answer rate (close domain q) w/o tool": [close_domain_without_tool_rate],
    "Answer rate (close domain q) w. tool (no history)": [close_domain_with_tool_no_history_rate],
    "Answer rate (close domain q) w. tool (history)": [close_domain_with_tool_history_rate]
})

summary

All four experiments, numbered

In [ ]:
experiments = pd.DataFrame({
    "#": [1, 2, 3, 4],
    "Experiment": [
        "Tool domain input, tool present",
        "Close domain input, tool absent",
        "Close domain input, tool present",
        "Tool domain input -> close domain input, tool present",
    ],
    "Conv": ["Conv1", "Conv2", "Conv3", "Conv4"],
    "Tool available": [True, False, True, True],
    "History": [False, False, False, True],
    "Correct (%)": [
        df["Conv1 Correct"].mean() * 100,
        df["Conv2 Correct"].mean() * 100,
        df["Conv3 Correct"].mean() * 100,
        df["Conv4 Correct"].mean() * 100,
    ],
    "Tool used (%)": [
        df["Conv1 Tool Used"].mean() * 100,
        float("nan"),
        df["Conv3 Tool Used"].mean() * 100,
        df["Conv4 Tool Used"].mean() * 100,
    ],
})

experiments

Push to the shared Google Sheet — SAFE BY DEFAULT, writes nothing until you flip the flag

In [ ]:
# =====================================================================
#  SAFETY FLAG
#  While this is False, this cell writes NOTHING to the shared sheet.
#  Run everything, check the tables above, then set it to True and
#  re-run THIS CELL ONLY.
# =====================================================================
PUSH_TO_SHEET = False
# =====================================================================


SHEET_COLUMNS = [
    "Trial", "Tool Prompt", "Close Prompt",
    "Conv1 Tool Used", "Conv1 Response",
    "Conv2 Response",
    "Conv3 Tool Used", "Conv3 Response",
    "Conv4 Tool Used", "Conv4 Response",
    "Conv1 Correct", "Conv2 Correct", "Conv3 Correct", "Conv4 Correct",
]

missing = [c for c in SHEET_COLUMNS if c not in df.columns]
assert not missing, f"missing columns: {missing}"

df_sheet = df[SHEET_COLUMNS]          # drops "Close Answer" so the block
                                      # lines up with the other 7 domains

existing_rows = len(worksheet.get_all_values())
start_row = existing_rows + 1
experiment_name = f"{TOOL_DOMAIN} / {CLOSE_DOMAIN} / {MODEL}"

print("target sheet :", worksheet.spreadsheet.title, "/", worksheet.title)
print("rows in use  :", existing_rows)
print("header row   :", start_row, "->", experiment_name)
print("data rows    :", start_row + 1, "to", start_row + 1 + len(df_sheet))
print("shape        :", df_sheet.shape, " (must be (50, 14))")
print()

if not PUSH_TO_SHEET:
    print("PUSH_TO_SHEET is False - NOTHING WAS WRITTEN.")
    print("Check the tables above. When you are happy:")
    print("  1. set PUSH_TO_SHEET = True")
    print("  2. re-run this cell only")
else:
    worksheet.update(
        range_name=f"A{start_row}",
        values=[[experiment_name]]
    )
    set_with_dataframe(
        worksheet,
        df_sheet,
        row=start_row + 1,
        col=1,
        include_index=False,
        include_column_header=True
    )
    worksheet.update(
        range_name=f"A{start_row + len(df_sheet) + 2}",
        values=[[""]]
    )
    print(f"WRITTEN: '{experiment_name}' starting at row {start_row}.")


Local CSV backup (files appear in the Colab file browser on the left)

In [ ]:
from google.colab import files

slug = CLOSE_DOMAIN.replace(" ", "_")

paths = {
    f"{slug}_raw_full.csv":    df,          # everything, incl. Close Answer
    f"{slug}_raw_sheet.csv":   df_sheet,    # exactly the 14 sheet columns
    f"{slug}_summary.csv":     summary,     # the one row for the summary sheet
    f"{slug}_experiments.csv": experiments, # the 4-experiment table
}

for name, frame in paths.items():
    frame.to_csv(name, index=False)
    print(f"saved {name:32s} {frame.shape}")

print()
print("downloading to your computer...")
for name in paths:
    files.download(name)

print()
print("ALL DONE.")
